In [ ]:
import os
import json
import base64
from datetime import datetime
from typing import List, Dict, Any, Optional, Union
from pathlib import Path
import logging
from abc import ABC, abstractmethod
from dataclasses import dataclass
from enum import Enum

# Configuration classes
class ModelProvider(Enum):
    GEMINI = "gemini"
    OPENAI = "openai"

@dataclass
class AnalysisConfig:
    """Configuration for construction site analysis."""
    model_provider: ModelProvider
    model_name: str
    api_key: str
    memory_file_path: str = "construction_memory.txt"
    max_images_per_request: int = 20
    max_file_size_mb: int = 15
    enable_detailed_logging: bool = True

# Abstract base class for AI providers
class AIProvider(ABC):
    """Abstract base class for AI model providers."""
    
    def __init__(self, config: AnalysisConfig):
        self.config = config
        self.logger = logging.getLogger(self.__class__.__name__)
        
    @abstractmethod
    def analyze_images(self, images: List[Dict[str, Any]], prompt: str) -> str:
        """Analyze images with the given prompt."""
        pass
    
    @abstractmethod
    def generate_structured_response(self, prompt: str) -> str:
        """Generate a structured response from a text prompt."""
        pass

# Gemini AI Provider
class GeminiProvider(AIProvider):
    """Google Gemini AI provider implementation."""
    
    def __init__(self, config: AnalysisConfig):
        super().__init__(config)
        try:
            from google import genai
            from google.genai import types
            self.genai = genai
            self.types = types
            self.client = genai.Client(api_key=config.api_key)
        except ImportError:
            raise ImportError("Please install google-genai: pip install google-genai")
    
    def analyze_images(self, images: List[Dict[str, Any]], prompt: str) -> str:
        """Analyze images using Gemini."""
        try:
            # Prepare image parts for Gemini
            image_parts = []
            
            for img_data in images:
                if img_data['method'] == 'file_upload':
                    # Upload file using Gemini File API
                    uploaded_file = self.client.files.upload(file=img_data['path'])
                    image_parts.append(uploaded_file)
                else:
                    # Use inline data
                    image_part = self.types.Part.from_bytes(
                        data=img_data['data'],
                        mime_type=img_data['mime_type']
                    )
                    image_parts.append(image_part)
            
            # Prepare content for API
            contents = [prompt] + image_parts
            
            # Call Gemini API
            response = self.client.models.generate_content(
                model=self.config.model_name,
                contents=contents,
            )
            
            return response.text
            
        except Exception as e:
            self.logger.error(f"Gemini analysis failed: {e}")
            raise
    
    def generate_structured_response(self, prompt: str) -> str:
        """Generate structured response using Gemini."""
        try:
            response = self.client.models.generate_content(
                model=self.config.model_name,
                contents=[prompt],
            )
            return response.text
        except Exception as e:
            self.logger.error(f"Gemini structured response failed: {e}")
            raise

# OpenAI Provider
class OpenAIProvider(AIProvider):
    """OpenAI provider implementation."""
    
    def __init__(self, config: AnalysisConfig):
        super().__init__(config)
        try:
            import openai
            self.client = openai.OpenAI(api_key=config.api_key)
        except ImportError:
            raise ImportError("Please install openai: pip install openai")
    
    def analyze_images(self, images: List[Dict[str, Any]], prompt: str) -> str:
        """Analyze images using OpenAI GPT-4 Vision."""
        try:
            # Prepare messages for OpenAI
            messages = [
                {
                    "role": "user",
                    "content": [
                        {"type": "text", "text": prompt}
                    ]
                }
            ]
            
            # Add images to message content
            for img_data in images:
                if img_data['method'] == 'base64':
                    image_content = {
                        "type": "image_url",
                        "image_url": {
                            "url": f"data:{img_data['mime_type']};base64,{img_data['base64_data']}"
                        }
                    }
                    messages[0]["content"].append(image_content)
                else:
                    # For file uploads, we need to convert to base64
                    with open(img_data['path'], 'rb') as f:
                        image_bytes = f.read()
                    base64_image = base64.b64encode(image_bytes).decode('utf-8')
                    image_content = {
                        "type": "image_url", 
                        "image_url": {
                            "url": f"data:{img_data['mime_type']};base64,{base64_image}"
                        }
                    }
                    messages[0]["content"].append(image_content)
            
            # Call OpenAI API
            response = self.client.chat.completions.create(
                model=self.config.model_name,
                messages=messages,
                max_tokens=4000
            )
            
            return response.choices[0].message.content
            
        except Exception as e:
            self.logger.error(f"OpenAI analysis failed: {e}")
            raise
    
    def generate_structured_response(self, prompt: str) -> str:
        """Generate structured response using OpenAI."""
        try:
            response = self.client.chat.completions.create(
                model=self.config.model_name,
                messages=[{"role": "user", "content": prompt}],
                max_tokens=2000
            )
            return response.choices[0].message.content
        except Exception as e:
            self.logger.error(f"OpenAI structured response failed: {e}")
            raise

# Image Processing Module
class ImageProcessor:
    """Handles image file processing and preparation."""
    
    def __init__(self, config: AnalysisConfig):
        self.config = config
        self.supported_formats = {'.jpg', '.jpeg', '.png', '.webp', '.heic', '.heif'}
        self.logger = logging.getLogger(self.__class__.__name__)
    
    def get_image_files(self, folder_path: str) -> List[str]:
        """Get all supported image files from folder."""
        folder = Path(folder_path)
        if not folder.exists():
            raise FileNotFoundError(f"Folder not found: {folder_path}")
        
        image_files = []
        for file_path in folder.iterdir():
            if file_path.is_file() and file_path.suffix.lower() in self.supported_formats:
                image_files.append(str(file_path))
        
        if not image_files:
            raise ValueError(f"No supported image files found in {folder_path}")
        
        image_files.sort()
        self.logger.info(f"Found {len(image_files)} image files")
        return image_files
    
    def prepare_images(self, image_files: List[str]) -> List[Dict[str, Any]]:
        """Prepare images for AI analysis."""
        prepared_images = []
        max_size_bytes = self.config.max_file_size_mb * 1024 * 1024
        
        for image_path in image_files[:self.config.max_images_per_request]:
            try:
                file_size = os.path.getsize(image_path)
                mime_type = self._get_mime_type(image_path)
                
                if file_size > max_size_bytes:
                    # Use file upload method for large files
                    image_data = {
                        'path': image_path,
                        'method': 'file_upload',
                        'mime_type': mime_type,
                        'size': file_size
                    }
                else:
                    # Use inline/base64 method for smaller files
                    with open(image_path, 'rb') as f:
                        image_bytes = f.read()
                    
                    image_data = {
                        'path': image_path,
                        'method': 'base64',
                        'data': image_bytes,
                        'base64_data': base64.b64encode(image_bytes).decode('utf-8'),
                        'mime_type': mime_type,
                        'size': file_size
                    }
                
                prepared_images.append(image_data)
                self.logger.info(f"Prepared image: {image_path} ({file_size} bytes)")
                
            except Exception as e:
                self.logger.error(f"Failed to prepare image {image_path}: {e}")
                continue
        
        return prepared_images
    
    def _get_mime_type(self, image_path: str) -> str:
        """Get MIME type from file extension."""
        ext = Path(image_path).suffix.lower()
        mime_type_map = {
            '.jpg': 'image/jpeg',
            '.jpeg': 'image/jpeg', 
            '.png': 'image/png',
            '.webp': 'image/webp',
            '.heic': 'image/heic',
            '.heif': 'image/heif'
        }
        return mime_type_map.get(ext, 'image/jpeg')

# Memory Management Module
class MemoryManager:
    """Handles project memory and historical data."""
    
    def __init__(self, memory_file_path: str):
        self.memory_file_path = memory_file_path
        self.logger = logging.getLogger(self.__class__.__name__)
    
    def read_memory(self) -> Dict[str, Any]:
        """Read historical construction data."""
        try:
            if os.path.exists(self.memory_file_path):
                with open(self.memory_file_path, 'r', encoding='utf-8') as f:
                    content = f.read().strip()
                    if content:
                        return json.loads(content)
            
            return self._get_default_memory()
            
        except (json.JSONDecodeError, FileNotFoundError) as e:
            self.logger.warning(f"Could not read memory file: {e}")
            return self._get_default_memory()
    
    def write_memory(self, memory_data: Dict[str, Any]) -> None:
        """Write updated memory data."""
        try:
            with open(self.memory_file_path, 'w', encoding='utf-8') as f:
                json.dump(memory_data, f, indent=2, ensure_ascii=False)
            self.logger.info(f"Memory updated: {self.memory_file_path}")
        except Exception as e:
            self.logger.error(f"Failed to write memory: {e}")
    
    def update_daily_report(self, memory_data: Dict[str, Any], date: str, 
                          analysis: str, images_count: int) -> Dict[str, Any]:
        """Update memory with daily report."""
        memory_data["total_days_analyzed"] += 1
        memory_data["daily_reports"][date] = {
            "summary": analysis[:500] + "..." if len(analysis) > 500 else analysis,
            "full_analysis": analysis,
            "images_analyzed": images_count,
            "timestamp": datetime.now().isoformat()
        }
        return memory_data
    
    def add_milestone(self, memory_data: Dict[str, Any], date: str, 
                     description: str, progress: float) -> Dict[str, Any]:
        """Add a milestone to memory."""
        if progress > 10:  # Only significant progress
            milestone = {
                "date": date,
                "description": description,
                "progress_percentage": progress
            }
            memory_data["key_milestones"].append(milestone)
        return memory_data
    
    def _get_default_memory(self) -> Dict[str, Any]:
        """Get default memory structure."""
        return {
            "project_start_date": None,
            "total_days_analyzed": 0,
            "daily_reports": {},
            "key_milestones": [],
            "current_phase": "Unknown",
            "overall_progress_percentage": 0
        }

# Main Construction Site Analyzer
class ConstructionSiteAnalyzer:
    """Main construction site analyzer that orchestrates all components."""
    
    def __init__(self, config: AnalysisConfig):
        self.config = config
        self.logger = self._setup_logging()
        
        # Initialize components
        self.ai_provider = self._create_ai_provider()
        self.image_processor = ImageProcessor(config)
        self.memory_manager = MemoryManager(config.memory_file_path)
    
    def _setup_logging(self) -> logging.Logger:
        """Setup logging configuration."""
        if self.config.enable_detailed_logging:
            logging.basicConfig(
                level=logging.INFO,
                format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
            )
        return logging.getLogger(self.__class__.__name__)
    
    def _create_ai_provider(self) -> AIProvider:
        """Create appropriate AI provider based on configuration."""
        if self.config.model_provider == ModelProvider.GEMINI:
            return GeminiProvider(self.config)
        elif self.config.model_provider == ModelProvider.OPENAI:
            return OpenAIProvider(self.config)
        else:
            raise ValueError(f"Unsupported provider: {self.config.model_provider}")
    
    def analyze_construction_site(self, folder_path: str) -> Dict[str, Any]:
        """Analyze construction site images and generate comprehensive report."""
        try:
            today = datetime.now().strftime("%Y-%m-%d")
            
            # Read existing memory
            memory_data = self.memory_manager.read_memory()
            if not memory_data.get("project_start_date"):
                memory_data["project_start_date"] = today
            
            # Process images
            image_files = self.image_processor.get_image_files(folder_path)
            prepared_images = self.image_processor.prepare_images(image_files)
            
            if not prepared_images:
                raise ValueError("No images could be prepared for analysis")
            
            # Create analysis prompt
            prompt = self._create_analysis_prompt(memory_data, today)
            
            # Analyze with AI
            self.logger.info(f"Analyzing with {self.config.model_provider.value}")
            analysis_text = self.ai_provider.analyze_images(prepared_images, prompt)
            
            # Update memory with analysis
            memory_data = self.memory_manager.update_daily_report(
                memory_data, today, analysis_text, len(image_files)
            )
            
            # Generate structured progress report
            progress_data = self._generate_progress_report(analysis_text, memory_data)
            
            # Update memory with progress data
            memory_data = self._update_memory_with_progress(memory_data, progress_data, today)
            
            # Save updated memory
            self.memory_manager.write_memory(memory_data)
            
            # Return results
            return self._format_results(today, len(image_files), analysis_text, 
                                      progress_data, memory_data)
            
        except Exception as e:
            self.logger.error(f"Analysis failed: {e}")
            return {
                "analysis_date": datetime.now().strftime("%Y-%m-%d"),
                "error": str(e),
                "status": "failed"
            }
    
    def _create_analysis_prompt(self, memory_data: Dict[str, Any], today: str) -> str:
        """Create comprehensive analysis prompt."""
        context = self._build_context_from_memory(memory_data)
        
        return f"""
You are an expert construction project manager analyzing construction site images taken on {today}.

CONTEXT:
- Current project phase: {memory_data.get('current_phase', 'Unknown')}
- Total days analyzed: {memory_data.get('total_days_analyzed', 0)}
- Overall progress: {memory_data.get('overall_progress_percentage', 0)}%
- Project start: {memory_data.get('project_start_date', 'Unknown')}

{context}

ANALYSIS REQUIREMENTS:
1. Daily Activities: What construction work was performed today?
2. Progress Made: Specific accomplishments and completions
3. Key Observations: Notable changes, installations, issues
4. Safety Assessment: Safety practices and concerns
5. Equipment/Personnel: Visible equipment and workforce
6. Quality Control: Work quality assessment
7. Weather Impact: How conditions affected work
8. Timeline: Project schedule status

Provide detailed, professional analysis comparing with previous progress where available.
Focus on concrete observations and measurable progress indicators.
"""
    
    def _build_context_from_memory(self, memory_data: Dict[str, Any]) -> str:
        """Build context string from memory data."""
        context = ""
        if memory_data.get("daily_reports"):
            recent_reports = list(memory_data["daily_reports"].items())[-3:]
            context = "Recent progress:\n"
            for date, report in recent_reports:
                context += f"- {date}: {report.get('summary', 'No summary')}\n"
        return context
    
    def _generate_progress_report(self, analysis_text: str, memory_data: Dict[str, Any]) -> Dict[str, Any]:
        """Generate structured progress report."""
        progress_prompt = f"""
Based on this construction analysis, provide a structured assessment:

ANALYSIS: {analysis_text}

PROJECT DATA:
- Days analyzed: {memory_data['total_days_analyzed']}
- Previous progress: {memory_data.get('overall_progress_percentage', 0)}%
- Current phase: {memory_data.get('current_phase', 'Unknown')}

Respond with ONLY valid JSON in this exact format:
{{
    "current_phase": "Phase name",
    "overall_progress_percentage": 50,
    "daily_progress_percentage": 5,
    "key_accomplishments": ["accomplishment 1", "accomplishment 2"],
    "identified_issues": ["issue 1", "issue 2"],
    "next_phase_indicators": ["indicator 1", "indicator 2"],
    "estimated_timeline_status": "On Schedule",
    "critical_path_items": ["item 1", "item 2"]
}}
"""
        
        try:
            response = self.ai_provider.generate_structured_response(progress_prompt)
            return json.loads(response)
        except (json.JSONDecodeError, Exception) as e:
            self.logger.warning(f"Could not parse progress data: {e}")
            return self._get_default_progress_data(memory_data)
    
    def _get_default_progress_data(self, memory_data: Dict[str, Any]) -> Dict[str, Any]:
        """Get default progress data structure."""
        return {
            "current_phase": memory_data.get("current_phase", "Analysis in Progress"),
            "overall_progress_percentage": memory_data.get("overall_progress_percentage", 0),
            "daily_progress_percentage": 0,
            "key_accomplishments": [],
            "identified_issues": [],
            "next_phase_indicators": [],
            "estimated_timeline_status": "Unknown", 
            "critical_path_items": []
        }
    
    def _update_memory_with_progress(self, memory_data: Dict[str, Any], 
                                   progress_data: Dict[str, Any], today: str) -> Dict[str, Any]:
        """Update memory with progress data."""
        memory_data["current_phase"] = progress_data.get("current_phase", memory_data.get("current_phase"))
        memory_data["overall_progress_percentage"] = progress_data.get("overall_progress_percentage", 0)
        
        # Add milestone if significant progress
        memory_data = self.memory_manager.add_milestone(
            memory_data, today, 
            f"Progress in {progress_data.get('current_phase', 'construction')}",
            progress_data.get("daily_progress_percentage", 0)
        )
        
        return memory_data
    
    def _format_results(self, today: str, images_count: int, analysis_text: str,
                       progress_data: Dict[str, Any], memory_data: Dict[str, Any]) -> Dict[str, Any]:
        """Format final results."""
        return {
            "analysis_date": today,
            "model_used": f"{self.config.model_provider.value}/{self.config.model_name}",
            "images_processed": images_count,
            "daily_description": analysis_text,
            "progress_report": progress_data,
            "updated_memory": {
                "project_start_date": memory_data["project_start_date"],
                "total_days_analyzed": memory_data["total_days_analyzed"],
                "current_phase": memory_data["current_phase"],
                "overall_progress_percentage": memory_data["overall_progress_percentage"],
                "recent_milestones": memory_data["key_milestones"][-5:] if memory_data["key_milestones"] else []
            },
            "status": "success"
        }

# Factory functions for easy usage
def create_gemini_analyzer(api_key: str, model_name: str = "gemini-2.5-pro", 
                          memory_file: str = "construction_memory.txt") -> ConstructionSiteAnalyzer:
    """Create analyzer configured for Gemini."""
    config = AnalysisConfig(
        model_provider=ModelProvider.GEMINI,
        model_name=model_name,
        api_key=api_key,
        memory_file_path=memory_file
    )
    return ConstructionSiteAnalyzer(config)

def create_openai_analyzer(api_key: str, model_name: str = "gpt-4o", 
                          memory_file: str = "construction_memory.txt") -> ConstructionSiteAnalyzer:
    """Create analyzer configured for OpenAI."""
    config = AnalysisConfig(
        model_provider=ModelProvider.OPENAI,
        model_name=model_name,
        api_key=api_key,
        memory_file_path=memory_file
    )
    return ConstructionSiteAnalyzer(config)

# Convenience function
def analyze_construction_progress(provider: str, api_key: str, folder_path: str,
                                model_name: str = None, memory_file: str = "construction_memory.txt") -> Dict[str, Any]:
    """
    Analyze construction progress with specified provider.
    
    Args:
        provider: "gemini" or "openai"
        api_key: API key for the provider
        folder_path: Path to images folder
        model_name: Optional model name (uses defaults if None)
        memory_file: Path to memory file
    """
    if provider.lower() == "gemini":
        model_name = model_name or "gemini-2.5-pro"
        analyzer = create_gemini_analyzer(api_key, model_name, memory_file)
    elif provider.lower() == "openai":
        model_name = model_name or "gpt-4o"
        analyzer = create_openai_analyzer(api_key, model_name, memory_file)
    else:
        raise ValueError(f"Unsupported provider: {provider}")
    
    return analyzer.analyze_construction_site(folder_path)

# Example usage
if __name__ == "__main__":
    # Example with Gemini
    gemini_result = analyze_construction_progress(
        provider="gemini",
        api_key="your-gemini-api-key",
        folder_path="/path/to/images",
        model_name="gemini-2.5-pro"
    )
    
    # Example with OpenAI
    openai_result = analyze_construction_progress(
        provider="openai", 
        api_key="your-openai-api-key",
        folder_path="/path/to/images",
        model_name="gpt-4o"
    )
    
    print(json.dumps(gemini_result, indent=2))